# Fluxo básico de RAG GoBR

O notebook abre os índices salvos por `indexacao_explicacoes.ipynb` (descrições de `dados/entradas/v1/*.txt`) e `indexacao_manual.ipynb` (trechos extraídos do PDF). A busca usa os índices persistidos e lê os arquivos `.gobr` originais. O Qwen recebe os exemplos e as páginas do manual para propor uma regra.

Use GPU no Colab. Execute primeiro os notebooks de indexação. Se eles estiverem em sessões separadas, copie suas pastas de cache para esta sessão. O README mostra como transferi-las.

In [ ]:
%pip install -q "llama-index-core>=0.13,<0.15" "llama-index-embeddings-huggingface>=0.7,<0.8" "sentence-transformers>=3,<6" "transformers>=4.41,<5" "langgraph>=1,<2"

## 1. Abrir os dois índices

Por padrão, a célula procura um índice de cada tipo em `gobr_index_cache`, ao lado do repositório. Se houver várias versões ou se o cache estiver em outra pasta, indique abaixo o caminho de cada índice. Use a pasta que contém `manifesto.json`.

In [ ]:
import json
import os
from pathlib import Path
import subprocess

import torch
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Preencha somente se quiser selecionar uma pasta específica.
PASTA_EXPLICACOES = None
PASTA_MANUAL = None

pasta_padrao = Path.cwd() if Path("dados/referencias").is_dir() else Path("projeto-sistemas-multiagentes")
REPO_DIR = Path(os.environ.get("GOBR_REPO_DIR", pasta_padrao)).expanduser().resolve()
if not REPO_DIR.exists() and "GOBR_REPO_DIR" not in os.environ:
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/buenofgustavo/projeto-sistemas-multiagentes.git", str(REPO_DIR),
    ], check=True)
if not (REPO_DIR / "dados/referencias").is_dir():
    raise FileNotFoundError(f"Arquivos GoBR ausentes em {REPO_DIR / 'dados/referencias'}")

CACHE = REPO_DIR.parent / "gobr_index_cache"


def pasta_indice(caminho, padrao, variavel):
    manifestos = [Path(caminho).expanduser() / "manifesto.json"] if caminho else sorted(CACHE.glob(padrao))
    if not manifestos or not manifestos[0].is_file():
        raise FileNotFoundError(
            f"Índice ausente em {CACHE / padrao}. Execute o notebook de indexação "
            f"e copie o cache para esta sessão ou configure {variavel}."
        )
    if len(manifestos) > 1:
        opcoes = "\n".join(str(arquivo.parent) for arquivo in manifestos)
        raise ValueError(f"Escolha um índice em {variavel}:\n{opcoes}")
    return manifestos[0].parent


pasta_explicacoes = pasta_indice(PASTA_EXPLICACOES, "explicacoes/*/manifesto.json", "PASTA_EXPLICACOES")
pasta_manual = pasta_indice(PASTA_MANUAL, "manual/*/indice/manifesto.json", "PASTA_MANUAL")

config_explicacoes = json.loads((pasta_explicacoes / "manifesto.json").read_text(encoding="utf-8"))
config_manual = json.loads((pasta_manual / "manifesto.json").read_text(encoding="utf-8"))
modelo_embedding = config_explicacoes["modelo"]
if config_manual["modelo_embedding"] != modelo_embedding:
    raise ValueError("Selecione índices criados com o mesmo modelo de embeddings.")

gpu = torch.cuda.is_available()
precisao = torch.float16 if gpu else torch.float32
embedding = HuggingFaceEmbedding(
    model_name=modelo_embedding,
    device="cuda" if gpu else "cpu",
    max_length=max(config_explicacoes["execucao"]["max_length"], config_manual["comprimento_modelo"]),
    model_kwargs={"torch_dtype": precisao},
)
indice_explicacoes = load_index_from_storage(
    StorageContext.from_defaults(persist_dir=str(pasta_explicacoes)), embed_model=embedding,
)
indice_manual = load_index_from_storage(
    StorageContext.from_defaults(persist_dir=str(pasta_manual)), embed_model=embedding,
)

## 2. Carregar o Qwen

Execute esta célula uma vez por sessão do Colab, depois de abrir os índices.

In [ ]:
from transformers import pipeline

gerador = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    device=0 if gpu else -1,
    torch_dtype=precisao,
)

## 3. Montar o fluxo de busca e geração

O LangGraph busca dois exemplos e três trechos do manual. O Qwen recebe o pedido e essas fontes, e deve citar as referências usadas e indicar os dados que faltam para escrever a regra.

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class Estado(TypedDict, total=False):
    pedido: str
    contexto: str
    fontes: list[str]
    resposta: str


def buscar_contexto(estado: Estado):
    pedido = estado["pedido"].strip()
    if not pedido:
        raise ValueError("Informe uma solicitação comercial.")

    contexto, fontes = [], []
    for hit in indice_explicacoes.as_retriever(similarity_top_k=2).retrieve(pedido):
        dados = hit.node.metadata
        fonte = f"{dados['fonte_id']} ({dados['arquivo']})"
        codigo = (REPO_DIR / dados["arquivo"]).read_text(encoding="utf-8-sig")
        fontes.append(fonte)
        contexto.append(f"[{fonte}]\n{hit.node.text}\n{codigo}")

    for hit in indice_manual.as_retriever(similarity_top_k=3).retrieve(pedido):
        dados = hit.node.metadata
        paginas = ", ".join(map(str, dados["paginas_pdf"]))
        fonte = f"{dados['chunk_id']} ({dados['fonte']}, páginas PDF {paginas})"
        fontes.append(fonte)
        contexto.append(f"[{fonte}]\n{hit.node.text}")

    return {"contexto": "\n\n".join(contexto), "fontes": fontes}


def gerar_resposta(estado: Estado):
    if not estado["contexto"]:
        return {"resposta": "A busca não encontrou fontes para elaborar a regra."}

    mensagens = [
        {"role": "system", "content": ("""
            Você escreve regras na linguagem GoBR (Govendas Business Rules). Sua tarefa é transformar o pedido do usuário em uma parametrização para um arquivo .gobr. Responda em português.
              - Escopo: gere apenas as regras solicitadas e as definições necessárias para elas funcionarem. Preserve os nomes, IDs, valores, condições e exceções do pedido.
              - Uso das fontes: consulte o manual para confirmar a sintaxe e use os exemplos recuperados como referência de estrutura. Aproveite um dado de exemplo apenas se ele identificar de forma inequívoca uma entidade mencionada no pedido. Descarte regras, valores padrão e restrições que apareçam apenas nas referências.
              - Revisão: confira se cada regra tem o escopo, as condições e a ação solicitados, se as referências entre blocos estão definidas e se cada bloco está completo. Uma exceção ou regra residual deve ter uma condição que delimite sua aplicação. Use somente comandos e grafia confirmados nas fontes. Gere exclusivamente GoBR.
              - Lacunas: se faltar um dado indispensável, houver conflito entre as fontes ou a sintaxe necessária não estiver documentada no contexto, faça uma pergunta objetiva antes de gerar a regra. Evite preencher lacunas por suposição.
            Quando houver informação suficiente, entregue um único bloco de código marcado como gobr. Depois do bloco, cite apenas os IDs dos exemplos e as páginas do manual efetivamente utilizados.
        """)},
        {"role": "user", "content": f"Pedido:\n{estado['pedido']}\n\nFontes:\n{estado['contexto']}"},
    ]
    saida = gerador(mensagens, max_new_tokens=768, do_sample=False, return_full_text=False)
    return {"resposta": saida[0]["generated_text"].strip()}


fluxo = StateGraph(Estado)
fluxo.add_node("buscar_contexto", buscar_contexto)
fluxo.add_node("gerar_resposta", gerar_resposta)
fluxo.add_edge(START, "buscar_contexto")
fluxo.add_edge("buscar_contexto", "gerar_resposta")
fluxo.add_edge("gerar_resposta", END)
rag = fluxo.compile()

## 4. Fazer uma pergunta

Para cada nova pergunta, altere `pedido` e execute só a célula abaixo. Ela reutiliza os índices e o Qwen carregados na sessão. Confira a proposta contra as referências antes de aplicar o código.

In [ ]:
pedido = "Para a vendedora Yara, agendar clientes da curva A a cada 3 dias."

resultado = rag.invoke({"pedido": pedido})
print(resultado["resposta"])
print("\nFontes recuperadas")
for fonte in resultado["fontes"]:
    print("-", fonte)